In [1]:
import os
import sys
from pathlib import Path
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))

In [2]:
from SysSimX.core.config import load_config
from SysSimX.components.fmu_component import FMUComponent
from SysSimX.components.opensim_pendulum import OpenSimPendulum
from SysSimX.core.scheduler import run_controlled_pendulum

In [3]:
cfg_path = repo_root / 'SysSimX' / 'demos' / 'configs' / 'demo_hybrid.yaml'

# Check if the file exists
if not cfg_path.is_file():
    raise FileNotFoundError(f"The configuration file was not found: {cfg_path}")

cfg = load_config(str(cfg_path))

print("Configuration loaded successfully:")
for key, value in cfg.items():
    print(f"{key}: {value}")
    for subkey, subvalue in value.items():
        print(f"  {subkey}: {subvalue}")
        if isinstance(subvalue, dict):
            for subsubkey, subsubvalue in subvalue.items():
                print(f"    {subsubkey}: {subsubvalue}")

Configuration loaded successfully:
step: {'t0': 0.0, 'tf': 10.0, 'h': 0.001}
  t0: 0.0
  tf: 10.0
  h: 0.001
fmus: {'Reference': {'path': '/home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/Reference.fmu', 'inputs': {}, 'outputs': {'q_ref': 'q_ref'}}, 'SensorRef': {'path': '/home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/AngleEncoder.fmu', 'inputs': {'q': 'q'}, 'outputs': {'U_q': 'U_q'}}, 'SensorState': {'path': '/home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/AngleEncoder.fmu', 'inputs': {'q': 'q'}, 'outputs': {'U_q': 'U_q'}}, 'PID': {'path': '/home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/PID_Continuous.fmu', 'inputs': {'ref': 'ref', 'y': 'y'}, 'outputs': {'u': 'u'}}, 'Drive': {'path': '/home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/Drive.fmu', 'inputs': {'u_control': 'u_control', 'omega': 'omega'}, 'outputs': {'torque': 'torque', 'omega': 'omega'}}}
  Reference: {'path': '/home/flo/repos/SystemSimulation/demos

In [4]:
t0, tf, h = cfg["step"]["t0"], cfg["step"]["tf"], cfg["step"]["h"]

print(f"Simulation start time: {t0}")
print(f"Simulation end time: {tf}")
print(f"Simulation step size: {h}")

Simulation start time: 0.0
Simulation end time: 10.0
Simulation step size: 0.001


In [5]:
# Build FMU components
ref = FMUComponent('ref', cfg['fmus']['Reference']['path'],
                   inputs=cfg['fmus']['Reference']['inputs'],
                   outputs=cfg['fmus']['Reference']['outputs'])
sref  = FMUComponent("sref",  cfg["fmus"]["SensorRef"]["path"],
                    inputs=cfg["fmus"]["SensorRef"]["inputs"],
                    outputs=cfg["fmus"]["SensorRef"]["outputs"])
sstate= FMUComponent("sstate",cfg["fmus"]["SensorState"]["path"],
                    inputs=cfg["fmus"]["SensorState"]["inputs"],
                    outputs=cfg["fmus"]["SensorState"]["outputs"])
pid   = FMUComponent("pid",   cfg["fmus"]["PID"]["path"],
                    inputs=cfg["fmus"]["PID"]["inputs"],
                    outputs=cfg["fmus"]["PID"]["outputs"])
drive = FMUComponent("drive", cfg["fmus"]["Drive"]["path"],
                    inputs=cfg["fmus"]["Drive"]["inputs"],
                    outputs=cfg["fmus"]["Drive"]["outputs"])
plant = OpenSimPendulum()

for component in [ref, sref, sstate, pid, drive, plant]:
    component.initialize(t0)

In [6]:
# Simple logger
rows = []
def log(t, signals):
    rows.append({"t": t, **signals})

In [7]:
run_controlled_pendulum(ref=ref, sensor_ref=sref, sensor_state=sstate,
                        pid=pid, drive=drive, plant=plant,
                        t0=t0, tf=tf, h=h, logger=log)

In [9]:
for key, value in rows[-1].items():
    print(f"{key:<11}: {value:.4f}")

t          : 9.9990
q_ref      : -0.0000
q_state    : -0.1814
omega_state: 1.3836
U_q_ref    : 1.4941
U_q_state  : 1.3059
u_pid      : 0.1921
torque     : 2.7231


In [10]:
import numpy as np
t_vals = [row['t'] for row in rows]
q_ref_vals = [np.rad2deg(row['q_ref']) for row in rows]
q_state_vals = [np.rad2deg(row['q_state']) for row in rows]

In [11]:
# Create a simple plotly figure for the q_ref and q_state over time
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_vals, y=q_ref_vals, mode='lines',
                            name='Reference Angle (q_ref)', line=dict(color='white', dash='dash')))
fig.add_trace(go.Scatter(x=t_vals, y=q_state_vals, mode='lines',
                            name='State Angle (q_state)', line=dict(color='red')))
fig.update_layout(title='Pendulum Angle Tracking',
                  xaxis_title='Time (s)',
                  yaxis_title='Angle (degrees)',
                  legend_title='Legend',
                  template='plotly_dark')
fig.show()
